# HumanAI Detect — Veri Toplama (Colab)

**Amaç:** `ai_raw` ve `ai_humanized` etiketli metinleri Colab T4 GPU üzerinde
`Qwen/Qwen2.5-7B-Instruct` modeli ile üretmek.

**Adımlar:**
1. GPU kontrolü
2. Google Drive bağla
3. Repo'yu klonla, bağımlılıkları kur
4. `ai_raw` üret
5. `ai_humanized` üret
6. Sonuçları Drive'a kopyala

In [ ]:
# 1. GPU kontrolü
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Google Drive bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3a. Repo klonla (GitHub URL'ini kendi repo'nla değiştir)
GITHUB_REPO = 'https://github.com/KULLANICI_ADINIZ/humanai-detect.git'  # <- BURAYA KENDİ REPO URL'İNİ YAZ
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect

In [ ]:
# 3b. Bağımlılıkları kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

In [ ]:
# 3c. .env oluştur (API key gerekmez, transformers lokal çalışır)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 3d. target_count_per_class ayarla (test için 50, gerçek run için 3000)
import yaml, re

TARGET = 3000  # <- İstediğin sayıya ayarla (test için 50, gerçek run için 3000)

with open('configs/data_sources.yaml', 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(
    r'(target_count_per_class:\s*)\d+',
    f'target_count_per_class: {TARGET}',
    content
)
with open('configs/data_sources.yaml', 'w', encoding='utf-8') as f:
    f.write(content)

print(f'target_count_per_class = {TARGET}')

In [ ]:
# 4. ai_raw üret (Qwen2.5-7B-Instruct, 4-bit quantized, T4 GPU)
# Model ilk yüklemede ~5 dk sürer, sonraki çağrılar hızlı
!python scripts/collect_data.py --label ai_raw

In [ ]:
# 5. ai_humanized üret
!python scripts/collect_data.py --label ai_humanized

In [ ]:
# 6. Sonuçları Google Drive'a kopyala
import shutil, os
from pathlib import Path

DRIVE_OUTPUT = '/content/drive/MyDrive/humanai_detect_data'
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

for label in ['ai_raw', 'ai_humanized']:
    src = Path(f'data/raw/{label}/{label}.jsonl')
    if src.exists():
        dst = Path(DRIVE_OUTPUT) / f'{label}.jsonl'
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(src))
        print(f'{label}: {lines} ornek -> {dst}')
    else:
        print(f'{label}: dosya bulunamadi!')

print('\nDrive kopyalama tamamlandi.')

## Drive'dan Yerel Makineye Aktarım

Colab bittikten sonra:
1. `Google Drive > humanai_detect_data/` klasörünü aç
2. `ai_raw.jsonl` ve `ai_humanized.jsonl` dosyalarını indir
3. Yerel projede şuraya koy:
   - `data/raw/ai_raw/ai_raw.jsonl`
   - `data/raw/ai_humanized/ai_humanized.jsonl`
4. Sonra `preprocess.py`, `build_features.py`, `train_model.py` sırasıyla çalıştır